# Entra ID 3LO with AgentCore Gateway

This notebook demonstrates an agent on AgentCore Runtime connecting to tools via AgentCore Gateway with 3-Legged OAuth using Microsoft Entra ID.

## Architecture

```
User ──[ID Token]──► Agent Runtime ──► AgentCore Gateway ──[3LO]──► Microsoft Graph
                                              │
                                              │ If user not authorized
                                              ▼
                                        Auth URL returned
                                              │
User ◄────────────────────────────────────────┘
  │
  │ User consents at Entra ID
  ▼
AgentCore Token Vault stores tokens
```

## How Gateway 3LO Works

1. Agent calls Gateway tool (e.g., `MSGraph__getMyProfile`)
2. Gateway checks Token Vault for user's access token
3. If no token exists, Gateway returns an authorization URL
4. User visits URL, consents at Entra ID
5. Entra ID redirects to `defaultReturnUrl` with auth code
6. Gateway exchanges code for tokens, stores in Token Vault
7. Retry tool call - now succeeds with user's token

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import os
import uuid
import json
import boto3
import urllib.parse
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"

## Step 1: Configure Environment Variables

You need:
- **User App**: For authenticating users to the agent
- **Gateway 3LO App**: For accessing Microsoft Graph on behalf of users (must have Graph API permissions)

In [ ]:
# Entra ID Configuration
os.environ["ENTRA_TENANT_ID"] = "your-tenant-id"  # Replace
os.environ["ENTRA_USER_APP_ID"] = "your-user-app-client-id"  # Replace

# Gateway 3LO App - for accessing Microsoft Graph on behalf of users
# This app needs: User.Read, Calendars.Read permissions in Entra ID
os.environ["ENTRA_GATEWAY_3LO_APP_ID"] = "your-gateway-3lo-app-id"  # Replace
os.environ["ENTRA_GATEWAY_3LO_APP_SECRET"] = "your-gateway-3lo-app-secret"  # Replace

# OAuth callback URL - where user is redirected after consent
os.environ["OAUTH_CALLBACK_URL"] = "https://localhost:5173/oauth/callback"  # Replace with your app's callback

os.environ["ENTRA_USER_SCOPE"] = f"api://{os.environ['ENTRA_USER_APP_ID']}/.default openid profile"

## Step 2: Create 3LO Credential Provider for Gateway

This credential provider handles the OAuth authorization code flow.

In [ ]:
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

try:
    cp_response = agentcore_client.create_oauth2_credential_provider(
        name="entra-id-gateway-3lo",
        credentialProviderVendor="MicrosoftEntraId",
        oauth2ProviderConfigInput={
            "microsoftEntraIdProviderConfig": {
                "tenantId": os.environ["ENTRA_TENANT_ID"],
                "clientId": os.environ["ENTRA_GATEWAY_3LO_APP_ID"],
                "clientSecret": os.environ["ENTRA_GATEWAY_3LO_APP_SECRET"]
            }
        }
    )
    credential_provider_arn = cp_response['credentialProviderArn']
    callback_url = cp_response.get('callbackUrl', 'Check console')
    print(f"Created credential provider: {credential_provider_arn}")
    print(f"\n⚠️  Add this callback URL to your Entra ID app's redirect URIs: {callback_url}")
except agentcore_client.exceptions.ConflictException:
    print("Credential provider already exists")
    cp_info = agentcore_client.get_oauth2_credential_provider(name="entra-id-gateway-3lo")
    credential_provider_arn = cp_info['credentialProviderArn']
    print(f"Using existing: {credential_provider_arn}")

## Step 3: Create AgentCore Gateway

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)

try:
    gateway_response = agentcore_control.create_gateway(
        name="entra-id-3lo-gateway",
        description="Gateway with Entra ID 3LO for Microsoft Graph"
    )
    gateway_id = gateway_response['gatewayId']
    print(f"Created gateway: {gateway_id}")
except agentcore_control.exceptions.ConflictException:
    print("Gateway already exists")
    gateways = agentcore_control.list_gateways()
    for gw in gateways.get('gateways', []):
        if gw['name'] == 'entra-id-3lo-gateway':
            gateway_id = gw['gatewayId']
            print(f"Using existing: {gateway_id}")
            break

## Step 4: Create Gateway Target with 3LO Configuration

The Gateway target defines:
- **OpenAPI spec**: Defines the API endpoints that become MCP tools
- **Credential provider**: How to authenticate to the API (3LO in this case)

Key configuration for 3LO:
- `grantType`: `AUTHORIZATION_CODE` for 3LO
- `defaultReturnUrl`: Where to redirect user after consent
- `scopes`: Microsoft Graph permissions needed

In [ ]:
# Simplified Microsoft Graph OpenAPI spec
ms_graph_openapi = {
    "openapi": "3.0.0",
    "info": {"title": "Microsoft Graph", "version": "1.0"},
    "servers": [{"url": "https://graph.microsoft.com/v1.0"}],
    "paths": {
        "/me": {
            "get": {
                "operationId": "getMyProfile",
                "summary": "Get current user profile",
                "responses": {"200": {"description": "User profile"}}
            }
        },
        "/me/events": {
            "get": {
                "operationId": "getMyEvents",
                "summary": "Get current user calendar events",
                "parameters": [
                    {"name": "$top", "in": "query", "schema": {"type": "integer"}}
                ],
                "responses": {"200": {"description": "Calendar events"}}
            }
        }
    }
}

In [ ]:
try:
    target_response = agentcore_control.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="microsoft-graph-3lo",
        description="Microsoft Graph API with 3LO authentication",
        targetConfiguration={
            "mcpTargetConfiguration": {
                "openApiSchema": {
                    "inlineSchema": json.dumps(ms_graph_openapi)
                },
                "toolNamePrefix": "MSGraph"
            }
        },
        # Correct structure per AWS documentation
        credentialProviderConfigurations=[
            {
                "credentialProviderType": "OAUTH",
                "credentialProvider": {
                    "oauthCredentialProvider": {
                        "providerArn": credential_provider_arn,
                        "grantType": "AUTHORIZATION_CODE",
                        "defaultReturnUrl": os.environ["OAUTH_CALLBACK_URL"],
                        "scopes": ["User.Read", "Calendars.Read", "openid", "profile"]
                    }
                }
            }
        ],
        authorizationConfiguration={
            "authorizationType": "NONE"
        }
    )
    target_id = target_response['targetId']
    print(f"Created gateway target: {target_id}")
except agentcore_control.exceptions.ConflictException:
    print("Gateway target already exists")

In [ ]:
# Gateway MCP URL - this is what the agent connects to
gateway_mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/gateways/{gateway_id}/mcp"
print(f"Gateway MCP URL: {gateway_mcp_url}")

## Step 5: Create Agent that Connects to Gateway

In [ ]:
%%writefile agent_gateway.py
"""Agent connecting to AgentCore Gateway with 3LO"""
import os
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

GATEWAY_MCP_URL = os.environ.get("GATEWAY_MCP_URL")

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.1,
)

@app.entrypoint
def agent_handler(payload, context):
    prompt = payload.get("prompt", "hello")
    
    # User's token is passed through from inbound auth
    user_token = context.get("authorization_token", "")
    
    # Connect to Gateway - Gateway handles 3LO automatically
    headers = {"authorization": f"Bearer {user_token}"}
    mcp_client = MCPClient(lambda: streamablehttp_client(GATEWAY_MCP_URL, headers))
    
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=bedrock_model,
            tools=tools,
            system_prompt="""You are a helpful assistant with access to Microsoft Graph via Gateway.
            If a tool returns an authorization URL, tell the user to visit that URL to grant access."""
        )
        response = agent(prompt)
    
    return str(response)

if __name__ == "__main__":
    app.run()

## Step 6: Deploy Agent

In [ ]:
agent_runtime = Runtime()

agent_discovery_url = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/v2.0/.well-known/openid-configuration"

agent_config = agent_runtime.configure(
    entrypoint="agent_gateway.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="entra_id_gateway_agent",
    environment_variables={
        "GATEWAY_MCP_URL": gateway_mcp_url
    },
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": agent_discovery_url,
            "allowedAudience": [os.environ["ENTRA_USER_APP_ID"]]
        }
    }
)

agent_launch = agent_runtime.launch(local_build=True)
print(f"Agent deployed: {agent_launch.agent_id}")

## Step 7: Test Gateway 3LO Flow

In [ ]:
import msal
import webbrowser

authority = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}"
scopes = [os.environ["ENTRA_USER_SCOPE"]]

msal_app = msal.PublicClientApplication(
    client_id=os.environ["ENTRA_USER_APP_ID"],
    authority=authority,
)

result = msal_app.acquire_token_silent(scopes, account=None)
if not result:
    flow = msal_app.initiate_device_flow(scopes=scopes)
    print(flow["message"])
    webbrowser.open(flow["verification_uri"])
    result = msal_app.acquire_token_by_device_flow(flow)

bearer_token = result["access_token"]
print(f"Bearer Token Received: {bearer_token[:30]}...")

In [ ]:
import requests

escaped_agent_arn = urllib.parse.quote(agent_launch.agent_arn, safe='')
agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

session_id = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
}

# First call - may return authorization URL if user hasn't consented
response = requests.post(
    agent_url,
    data=json.dumps({"prompt": "Get my profile from Microsoft Graph"}),
    headers=headers
)
print(response.text)

## Using _meta to Customize 3LO Behavior

When calling Gateway tools, you can use the `_meta` field to:
- Override the return URL
- Force re-authentication

In [ ]:
# Example: Direct MCP call with _meta for 3LO customization
tool_call_with_meta = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "MSGraph__getMyProfile",
        "arguments": {},
        "_meta": {
            "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
                "oauthCredentialProvider": {
                    "returnUrl": "https://your-app.com/custom-callback",
                    "forceAuthentication": True  # Forces new consent even if token exists
                }
            }
        }
    }
}
print("Example _meta structure for 3LO customization:")
print(json.dumps(tool_call_with_meta, indent=2))

## Cleanup

In [ ]:
# Delete Agent
agentcore_control.delete_agent_runtime(agentRuntimeId=agent_launch.agent_id)
print("Deleted agent")

# Delete Gateway Target
agentcore_control.delete_gateway_target(
    gatewayIdentifier=gateway_id,
    targetIdentifier="microsoft-graph-3lo"
)
print("Deleted gateway target")

# Delete Gateway
agentcore_control.delete_gateway(gatewayIdentifier=gateway_id)
print("Deleted gateway")

# Delete Credential Provider
agentcore_client.delete_oauth2_credential_provider(name="entra-id-gateway-3lo")
print("Deleted credential provider")

## Conclusion

In this notebook we learned:

1. **Gateway 3LO Flow**: Gateway automatically handles OAuth authorization code flow
2. **Credential Provider Structure**: 
   ```python
   {
       "credentialProviderType": "OAUTH",
       "credentialProvider": {
           "oauthCredentialProvider": {
               "providerArn": "...",
               "grantType": "AUTHORIZATION_CODE",
               "defaultReturnUrl": "...",
               "scopes": [...]
           }
       }
   }
   ```
3. **_meta Field**: Customize OAuth behavior per-request
4. **Token Vault**: Gateway stores user tokens automatically

## Gateway vs Runtime MCP

| Aspect | Runtime MCP (Sample 2) | Gateway (Sample 3) |
|--------|------------------------|-------------------|
| OAuth Handling | Manual via URL elicitation | Built-in |
| Token Storage | Custom implementation | Token Vault |
| API Definition | Custom code | OpenAPI spec |
| Hosting | Self-managed container | Fully managed |